In [1]:
from typing import Any, List, Callable, Union
import h5py
import pickle
from pathlib import Path
import numpy as np
from scipy.stats import pearsonr
import torch
from enformer_pytorch import Enformer
from enformer_pytorch import from_pretrained
from enformer_pytorch.finetune import HeadAdapterWrapper
from transformers import get_scheduler
from torch.utils.data import TensorDataset, DataLoader

from torch.optim import AdamW
from tqdm.auto import tqdm

import sys
sys.path.append('../src')
import enformer_dataloader_np
from enformer_dataloader_np import NumpyDataModule

# use GPU 1:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

/home/rajesh/projects/hackathon/SAE_Hackathon/.venv/lib64/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Load Data

# Load Model

In [ ]:
#!/usr/bin/env python
import os
import argparse
import torch
import numpy as np
from tqdm import tqdm
import pytorch_lightning as pl
from enformer_pytorch import Enformer
from enformer_pytorch.finetune import HeadAdapterWrapper

# Import the NumpyDataModule class - assuming it's in your src directory
import sys
sys.path.append('src')
import enformer_dataloader_np
from enformer_dataloader_np import NumpyDataModule  # If your class definition is in this file

class EnformerWithEmbeddings(pl.LightningModule):
    def __init__(self, num_tracks=5313, target_layer='transformer.layers.5'):
        super().__init__()
        self.enformer = Enformer.from_pretrained('EleutherAI/enformer-official-rough')
        self.model = HeadAdapterWrapper(
            enformer=self.enformer,
            num_tracks=num_tracks,
            post_transformer_embed=False
        )
        
        self.target_layer = target_layer
        self.hook_store = {}
        self.set_hooks()
        
    def set_hooks(self):
        """Set up hooks to capture embeddings from target layer"""
        def get_activation(name):
            def hook(module, input, output):
                self.hook_store[name] = output.detach()
            return hook
        
        # Navigate to the target layer and register the hook
        layer_parts = self.target_layer.split('.')
        target = self.model.enformer
        for part in layer_parts:
            target = getattr(target, part)
        
        target.register_forward_hook(get_activation(self.target_layer))
        
    def get_embeddings(self, x):
        """Extract embeddings from the target layer and also return model predictions
        
        Returns:
            tuple: (embeddings, predictions) where embeddings are from the target layer
                  and predictions are the full model output
        """
        self.eval()
        with torch.no_grad():
            # Run a forward pass to trigger the hooks and get predictions
            predictions = self.model(x)
            # Return the captured embeddings and predictions
            return self.hook_store[self.target_layer], predictions
    
    def forward(self, x, target=None):
        preds = self.model(x)
        if target is None:
            return preds
        return self.model(seq=x, target=target)

# part of forward pass of SAE training
def process_dataset(data_loader, model, output_dir, device='cuda', max_batches=None):
    """Process all batches in a dataset and save embeddings to disk"""
    os.makedirs(output_dir, exist_ok=True)
    
    batch_count = 0
    with torch.no_grad():
        for batch_idx, (sequences, targets) in enumerate(tqdm(data_loader, desc="Processing Batches")):
            if max_batches is not None and batch_idx >= max_batches:
                break
                
            # Move data to device
            sequences = sequences.to(device)
            
            # Get embeddings and predictions
            embeddings, predictions = model.get_embeddings(sequences)
            
            # Save embeddings and predictions for each sample in the batch
            for i in range(sequences.shape[0]):
                # Create a unique identifier for this sample
                sample_id = batch_idx * data_loader.batch_size + i
            
            batch_count += 1
    
    print(f"Processed {batch_count} batches, saved embeddings to {output_dir}")


human_data = '/home/rajesh/projects/hackathon/SAE_Hackathon/data/enformer_data_npz/human'
mouse_data = '/home/rajesh/projects/hackathon/SAE_Hackathon/data/enformer_data_npz/mouse'
            
output_dir = '/home/rajesh/projects/hackathon/SAE_Hackathon/results/enformer_embeddings'
target_layer = 'conv_tower.5.2.to_attn_logits'
batch_size = 4
max_batches = None
gpu = True  

# Set device
device = 'cuda' if gpu and torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# Initialize the model
target_layer = 'conv_tower.5.2.to_attn_logits'
model = EnformerWithEmbeddings(target_layer=target_layer)
model = model.to(device)
model.eval()

# Create output directories
human_output_dir = os.path.join(output_dir, 'human')
mouse_output_dir = os.path.join(output_dir, 'mouse')

# Initialize data modules
human_data_module = NumpyDataModule(
    train_files_pattern="/home/rajesh/projects/hackathon/SAE_Hackathon/data/enformer_data_partitioned/human/train",
    val_files_pattern="/home/rajesh/projects/hackathon/SAE_Hackathon/data/enformer_data_partitioned/human/valid",
    test_files_pattern="/home/rajesh/projects/hackathon/SAE_Hackathon/data/enformer_data_partitioned/human/test",
    batch_size=batch_size
)

mouse_data_module = NumpyDataModule(
    train_files_pattern="/home/rajesh/projects/hackathon/SAE_Hackathon/data/enformer_data_partitioned/mouse/train",
    val_files_pattern="/home/rajesh/projects/hackathon/SAE_Hackathon/data/enformer_data_partitioned/mouse/valid",
    test_files_pattern="/home/rajesh/projects/hackathon/SAE_Hackathon/data/enformer_data_partitioned/mouse/test",
    batch_size=batch_size
)

# Setup data modules
human_data_module.setup()
mouse_data_module.setup()

# Get data loaders
human_loader = human_data_module.test_dataloader()
mouse_loader = mouse_data_module.test_dataloader()

# Process human dataset
print("Processing human dataset...")
process_dataset(human_loader, model, human_output_dir, device, max_batches)

# Process mouse dataset
print("Processing mouse dataset...")
process_dataset(mouse_loader, model, mouse_output_dir, device, max_batches)

print("Embedding extraction complete!")


# Load SAE

# Train Model

In [ ]:

import sys
sys.path.append('./src')
import argparse
import torch
from SAE_models import get_cfg, TopKSAE, VanillaSAE, JumpReLUSAE, BatchTopKSAE
from SAE_training import SAETraining
from torch.utils.data import DataLoader
import numpy as np
import json
import os
import yaml

# Simple argument parser to get config file path

config = 'configs/test_config.yaml'
override_config = None

# Load the main configuration file
if not os.path.exists(config):
    raise FileNotFoundError(f"Config file not found: {config}")
    
with open(config, 'r') as f:
    config_dict = yaml.safe_load(f)

# Load override configurations if specified
if override_config and os.path.exists(override_config):
    with open(override_config, 'r') as f:
        override_cfg = yaml.safe_load(f)
        config_dict.update(override_cfg)
    print(f"Loaded override configuration from {override_config}")

# Convert to Namespace for compatibility with existing code
args_namespace = argparse.Namespace()
for key, value in config_dict.items():
    setattr(args_namespace, key, value)

############################################################
################### 2. Training setup ######################
################### Not model specific #####################
############################################################

cfg = get_cfg(**vars(args_namespace))

trainer = SAETraining(cfg)


############################################################
################### 3. Data setup ##########################
################### Model specific #########################
############################################################

    # Set random seed for reproducibility
torch.manual_seed(cfg['seed'])
np.random.seed(cfg['seed'])



# Load input seqs, embeddings and model predictions (special Dataloader)
train_dl = EnformerDataloader(cfg['train_loader'])
#seqs, embed, pred = train_dl[0]

val_dl = EnformerDataloader(cfg['val_loader'])

test_dl  = EnformerDataloader(cfg['test_loader'])    

############################################################
################### 4. SAE Model setup #####################
################### Not Model specific #####################
############################################################


if cfg['sae_type'] == 'topk':
    model = TopKSAE(cfg)
elif cfg['sae_type'] == 'vanilla':
    model = VanillaSAE(cfg)
elif cfg['sae_type'] == 'jumprelu':
    model = JumpReLUSAE(cfg)
elif cfg['sae_type'] == 'batch_topk':
    model = BatchTopKSAE(cfg)

############################################################
################### 5. Training ############################
################### Not Model specific #####################
############################################################

final_model = trainer.train(model, train_dl, val_dl)


############################################################
################### 6. testing/validation ##################
################### Not Model specific #####################
############################################################

val_metrics = trainer.validate(val_dl)

print(val_metrics)
with open(cfg['outpath'] + f"{cfg['name']}_{cfg['seed']}_val_metrics.json", 'w') as f:
    json.dump(val_metrics, f)

test_metrics = trainer.test(test_dl)

print(test_metrics)
with open(cfg['outpath'] + f"{cfg['name']}_{cfg['seed']}_test_metrics.json", 'w') as f:
    json.dump(test_metrics, f)

